# Box Beam Bridge Walkthrough

A prestressed adjacent box-beam bridge carried through the same BrIM
architecture as the steel-girder slice: the ODOT **PSBD-1-25 /
PSBDD-1-25** standard designs supply the members, the L1 pipeline
re-derives the governing LRFD checks, the emit layer produces the tagged
Rhino model, and the same read-back gives the quantity estimate.

The bridge: a single 60 ft span of nine **CB27-48** composite boxes
(36 ft out-to-out) with the standard 6 in cast-in-place topping.

In [1]:
from civilpy.structural.odot import box_beam_design
from civilpy.structural.odot.box_beam_design import designs_for_box

design = box_beam_design("CB27-48", 60)
print(f"{design.box} @ {design.span} ft ({design.beam_type})")
print(f"  strands: {design.n_strands} total = {design.strands_2in} @ 2in"
      f" + {design.strands_4in} @ 4in + {design.strands_6in} @ 6in, "
      f"e = {design.e_beam} in")
print(f"  camber: {design.camber_d0} in release / {design.camber_d30} in"
      f" erection, bearing type {design.bearing_type}")
print("\ncataloged spans for CB27-48:",
      [d.span for d in designs_for_box("CB27-48")])

CB27-48 @ 60 ft (composite)
  strands: 24 total = 16 @ 2in + 8 @ 4in + 0 @ 6in, e = 10.83 in
  camber: 1.125 in release / 1.875 in erection, bearing type B2

cataloged spans for CB27-48: [40, 45, 50, 55, 60, 65, 70]


## 1. L1 verification of the standard design

`box_beam_line_checks` re-derives what the sheet's designers already
proved: HL-93 demands from the influence-line envelope with the
**adjacent-box** distribution factors (LRFD 4.6.2.2.2b/3c), elastic
shortening + approximate lump-sum losses (5.9.3), the transfer and
service stress checks (5.9.2.3, transfer evaluated at the 60-diameter
transfer length), and Strength I flexure (5.6.3). Railing and future
wearing surface enter as bridge totals shared across the nine boxes.

In [2]:
from civilpy.structural.box_beam_pipeline import box_beam_line_checks

checks = box_beam_line_checks("CB27-48", 60.0, 9,
                              barrier_klf=1.0,   # two SBR runs, bridge total
                              fws_klf=0.54)      # 0.015 ksf x 36 ft
print(checks.summary())
assert checks.all_ok

CB27-48 @ 60 ft (composite), 24 strands, e = 10.83 in:
  DF moment 0.295 / shear 0.458 (adjacent box, 4.6.2.2.2b/3c)
  losses: ES 12.2 + LT 24.8 ksi -> f_pe = 165.5 ksi
  PASS  transfer compression: D/C = 0.92 (5.9.2.3.1a)
  PASS  transfer tension: D/C = 0.93 (5.9.2.3.1b)
  PASS  service compression: D/C = 0.35 (5.9.2.3.2a)
  PASS  service III tension: D/C = 0.00 (5.9.2.3.2b)
  PASS  Strength I flexure: D/C = 0.67 (5.6.3.2.2)
  camber: 1.125 in release, 1.875 in erection (tabulated)


In [3]:
m = checks.midspan_moments
print("midspan moments per beam (kip-ft):")
for case, val in m.items():
    print(f"  {case:<8} {val:8.1f}")
print(f"\nservice stresses (ksi, compression +):")
for name, s in checks.stresses.items():
    print(f"  {name:<26} {s:7.3f}")

midspan moments per beam (kip-ft):
  sw          334.6
  topping     135.0
  dc2          50.0
  dw           27.0
  ll          398.9

service stresses (ksi, compression +):
  transfer_top_end            -0.444
  transfer_bot_end             2.379
  transfer_top_mid             0.249
  transfer_bot_mid             1.697
  service_top_permanent        0.740
  service_top_total            1.171
  service_bot_serviceIII       0.300


At the catalog's longest spans the fully-bonded transfer-tension
check runs just over 1.0 — the condition the sheet's **strand
debonding** exists to fix (the emit and checks assume full bonding,
conservative everywhere else).

## 2. The BrIM model

`box_beam_bridge_emit` ports the legacy box-beam writer to the emit
architecture: hollow-tube members (four wall prisms each), the strand
rows of the PSBDD pattern, diaphragms + tie rods at the standard
stations, bearing pads, and the composite topping — every object tagged,
one 515 member count per beam.

In [4]:
from civilpy.structural.rhino_box_bim import (
    BoxBridgeInput, box_beam_bridge_emit)
from civilpy.structural.rhino_bim import emit_to_json, pay_item_quantities

emit = box_beam_bridge_emit(BoxBridgeInput(
    box="CB27-48", span_ft=60.0, n_beams=9))
by_type = {}
for o in emit.objects:
    t = o.tags.get("bim.type")
    if t:
        by_type[t] = by_type.get(t, 0) + 1
print(by_type)

for item, rec in pay_item_quantities(emit).items():
    print(f"{item}  {rec['desc'][:52]:<52} {rec['qty']:>10,.1f} {rec['unit']}")

{'bridge': 1, 'box_beam': 36, 'tendon': 18, 'bearing': 18, 'diaphragm': 2, 'tie_rod': 2, 'deck': 1}
511E12100  Class QC2 concrete, superstructure (deck) [CONFIRM]        40.0 cy
515E10000  Prestressed concrete box beam member [CONFIRM]              9.0 ea
516E10000  Elastomeric bearing [CONFIRM]                              18.0 ea


In [5]:
with open("box_beam_emit.json", "w") as f:
    f.write(emit_to_json(emit))
print("wrote box_beam_emit.json for draw_bim_emit.py "
      "(same driver as the steel slice)")

wrote box_beam_emit.json for draw_bim_emit.py (same driver as the steel slice)


## 3. The MIDAS spoke

One line of beam elements per box, broken at the diaphragm stations
with transverse tie elements (the tie-rod / keyway load path), pin +
roller per line, and the same DC1/DC2/DW loads as the L1 checks — so
the two reconcile by construction on a simple span.

In [6]:
from civilpy.structural.box_beam_pipeline import structural_model_from_box

model = structural_model_from_box("CB27-48", 60.0, 9,
                                  barrier_klf=1.0, fws_klf=0.54)
girders = [e for e in model.elements.values() if e.role == "girder"]
ties = [e for e in model.elements.values() if e.role == "diaphragm"]
print(f"{len(model.nodes)} nodes, {len(girders)} girder elements, "
      f"{len(ties)} tie elements, {len(model.restraints)} restraints, "
      f"{len(model.beam_loads)} beam loads")
print("element section metadata:", girders[0].metadata)

36 nodes, 27 girder elements, 16 tie elements, 18 restraints, 81 beam loads
element section metadata: {'gdr.line': '1', 'gdr.family': 'box', 'section.area_in2': 713.8, 'section.i_in4': 66222, 'section.j_in4': 143505.1513671875}


---

## Where this leaves us

* **Standard design in, checks out**: the PSBDD-1-25 line re-derived and
  passing (losses, transfer/service stresses, Strength I flexure) with
  the adjacent-box distribution factors.
* **BrIM model**: tagged members / strands / tie rods / diaphragms /
  pads / topping, drawable by `draw_bim_emit.py` and read back into the
  same estimate as the steel slice.
* **MIDAS spoke**: the line-beam hub model with tie elements, loads
  matched to the L1 pipeline.

**Preliminary flags:** zero skew only (PSBD sheet 2/6 skew rules for
diaphragm/tie offsets pending); no shear-key geometry; strand debonding
not modeled (transfer checks conservative at max spans); railing on the
topping and the substructure hand-off reuse are the next steps.